In [ ]:
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import seaborn as sns
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info() #This won't show everything as I think the number of features is high, hence just a summary
print("\n\n--------------------------------------\n\n")
df.info(all, show_counts= all) #This in compare to regular df.info(), shows the dataset information

In [ ]:
# Task 4: Write your code here:
df.describe() # Showing the statistics information

In [ ]:
# Task 1: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
# Task 2: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
df

In [ ]:
# Task 4: Write your code here:
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  #Feature Scaling!!
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

#It is inbalanced!!

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target",axis=1)
y = df['Target']

In [ ]:
# Task 2,3,4,5: Write your code here:
#We will be using StratifiedKFold as our data is imbalanced!

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

#We will list the following metrics eventhough accuarcy and F1 score were asked!
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test  = torch.tensor(X_test.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test  = torch.tensor(y_test.values, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create Dataloaders to train and test data in batches
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Print dataset sizes
print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")



class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # First linear layer: input features -> hidden layer
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second linear layer: hidden layer -> hidden layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # Third linear layer: hidden layer -> hidden layer
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # Output layer: hidden layer -> number of classes (logits)
        self.layer4 = nn.Linear(hidden_dim, output_dim)

        # ReLU activation for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, x):
        # First hidden layer
        a1 = self.relu(self.layer1(x))

        # Second hidden layer
        a2 = self.relu(self.layer2(a1))

        # Third hidden layer
        a3 = self.relu(self.layer3(a2))

        # Output layer (raw scores / logits)
        output = self.layer4(a3)  # we said we'll use softmax, where is it? ¯\(ツ)/¯

        return output


#Training Loop
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.to(device)         # shape: (batch_size, num_features)
        y_batch = y_batch.to(device)         # shape: (batch_size,)

        # Forward pass (outputs are logits)
        outputs = model(X_batch)             # shape: (batch_size, num_classes)
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

        accuracy = accuracy_score(y_test, y_batch)
        precision = precision_score(y_test, y_batch, zero_division=0)
        recall = recall_score(y_test, y_batch, zero_division=0)
        f1 = f1_score(y_test, y_batch, zero_division=0)


        lr_accuracy.append(accuracy)
        lr_precision.append(precision)
        lr_recall.append(recall)
        lr_f1.append(f1)

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss



#Validiation Loop
def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.to(device)     # shape: (batch_size, num_features)
            y_batch = y_batch.to(device)     # shape: (batch_size,)

            # Forward pass
            outputs = model(X_batch)         # shape: (batch_size, num_classes)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # Apply Softmax to get probabilities
            probabilities = F.softmax(outputs, dim=1)

            # Pick the classes with highest probabilities
            predicted = torch.argmax(probabilities, dim=1)

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
input_dim = X_train.shape[1]   # Number of tabular features
hidden_dim = 14                # Design choice
output_dim = 189                 # P-2:......B:188

# Instantiate model
model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

num_epochs = 20
learning_rate = 0.001

# Define criterion (loss function) - using CrossEntropyLoss as model now outputs logits
criterion = nn.CrossEntropyLoss()
# Define optimizer
optimizer = AdamW(model.parameters(), learning_rate)


train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

print('Training Complete!')




In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
#Couldn't figure it out as my code is not working :(
#If the code was working properly then the golden feaure will be shown clearly!

In [ ]:
# Task Bonus: Write your code here: